# **Test Ligand Class**

In [ ]:
from src.protein import Protein

# Esempio di utilizzo
pdb_path = ".\\data\\raw\\P-L\\2011-2019\\2l7u\\2l7u_protein.pdb"
pdb_poket_path = ".\\data\\raw\\P-L\\2011-2019\\2l7u\\2l7u_pocket.pdb"

protein_pdb = Protein.from_pdb_code("2l7u")
protein_path = Protein.from_pdb_path(pdb_path)
pocket = Protein.from_pdb_path(pdb_poket_path)

protein_pdb.visualize_protein_3D()
protein_path.visualize_protein_3D()
pocket.visualize_protein_3D()

In [ ]:
print("proteina da PDB code:")
protein_pdb.visualize_protein_graph()
print("proteina da path:")
protein_path.visualize_protein_graph()
print("pocket:")
pocket.visualize_protein_graph()

In [ ]:
protein= Protein.from_pdb_code("1a08")
protein.visualize_protein_graph()
x, edge_index, edge_attr, pos = protein.get_pyg_graph()

print(f"x: {x.shape}, edge_index: {edge_index.shape}, edge_attr: {edge_attr.shape}, pos: {pos.shape}")

print(edge_index)



In [ ]:
print("Node features (x): [amminoacido embedding 7D + b_factor + res_name]")
print("[idrofobicità, volume sterico, Polarizzabilità, polarità, Potenziale Isuelettrico, Indice di Refrattività, Propensione per l'Elica alpha] [b_factor] [res_name]")
print (protein._get_node_features())

AMMINO_ACIDS_DICT = {
    'ALA': 0, 'ARG': 1, 'ASN': 2, 'ASP': 3, 'CYS': 4,
    'GLN': 5, 'GLU': 6, 'GLY': 7, 'HIS': 8, 'ILE': 9,
    'LEU': 10,'LYS': 11,'MET': 12,'PHE': 13,'PRO': 14,
    'SER': 15,'THR': 16,'TRP': 17,'TYR': 18,'VAL': 19
}

graph = protein.graph
for n, d in graph.nodes(data=True):
    meiler_vals = list(d['meiler']) 
    print(f"{n} - Meiler: ["
        f"h1:{meiler_vals[0]:.2f}, "
        f"h2:{meiler_vals[1]:.2f}, "
        f"h3:{meiler_vals[2]:.2f}, "
        f"h4:{meiler_vals[3]:.2f}, "
        f"h5:{meiler_vals[4]:.2f}, "
        f"h6:{meiler_vals[5]:.2f}, "
        f"h7:{meiler_vals[6]:.2f}] "
        f"- B-factor: {d['b_factor']:.2f}"
        f"- Residue Name: [{d['residue_name']}-{AMMINO_ACIDS_DICT.get(d['residue_name'], 99)}]")

In [ ]:
print(edge_attr)

for u, v, d in graph.edges(data=True):
    print(d)
    print(f"Edge ({u}, {v}) - Type: {d['kind']} - Distance: {d['distance']:.2f} Å")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print(pos)
def plot_protein_graph_3d(pos, edge_index, edge_attr=None):
    """
    Visualizza il grafo della proteina in 3D usando Matplotlib.
    pos: tensore [N, 3] delle coordinate
    edge_index: tensore [2, E] degli indici dei legami
    edge_attr: opzionale, tensore [E, 2] dove la seconda colonna è il flag peptide_bond
    """
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')

    # 1. DISEGNO DEI NODI
    # Usiamo una sfumatura basata sulla posizione nella sequenza (indice del nodo)
    colors = np.linspace(0, 1, pos.shape[0])
    scatter = ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], 
                         c=colors, cmap='coolwarm', s=60, edgecolors='black', depthshade=True)

    # 2. DISEGNO DEGLI ARCHI (Edges)
    # Iteriamo su edge_index. Nota: edge_index è bidirezionale, 
    # ne prendiamo solo metà o usiamo un set per non disegnare due volte la stessa linea.
    seen_edges = set()
    
    # Trasponiamo per iterare sulle coppie [src, dst]
    edges = edge_index.t().numpy()
    
    for i, (src, dst) in enumerate(edges):
        # Ordiniamo la coppia per evitare duplicati (u,v) == (v,u)
        edge = tuple(sorted((src, dst)))
        if edge in seen_edges:
            continue
        seen_edges.add(edge)

        # Coordinate dei due punti
        p1 = pos[src]
        p2 = pos[dst]

        # Definiamo stile e colore in base al tipo di legame (se disponibile)
        color = 'black'
        linewidth = 0.5
        alpha = 0.3
        
        if edge_attr is not None:
            is_peptide = edge_attr[i, 0].item() # La colonna 1 è il flag peptide
            if is_peptide == 1.0:
                color = 'lightblue'  
                linewidth = 2.5
                alpha = 0.8
        
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], 
                color=color, linewidth=linewidth, alpha=alpha)

    # Impostazioni grafiche
    ax.set_title('Grafo 3D Proteina: Backbone (Blu) e Contatti Spaziali (Grigio)')
    ax.set_xlabel('X (Å)')
    ax.set_ylabel('Y (Å)')
    ax.set_zlabel('Z (Å)')
    
    # Per mantenere le proporzioni reali
    max_range = np.array([pos[:,0].max()-pos[:,0].min(), 
                          pos[:,1].max()-pos[:,1].min(), 
                          pos[:,2].max()-pos[:,2].min()]).max() / 2.0
    mid_x = (pos[:,0].max()+pos[:,0].min()) * 0.5
    mid_y = (pos[:,1].max()+pos[:,1].min()) * 0.5
    mid_z = (pos[:,2].max()+pos[:,2].min()) * 0.5
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)

    plt.show()

# Esecuzione:
plot_protein_graph_3d(pos, edge_index, edge_attr)